# 11 — Regularização, BatchNorm, Dropout e callbacks

**EP · Data Science Knowledge Base | Deep Learning | Notebook 11/20**


**Pré-requisitos:** Python, NumPy, fundamentos de aprendizado supervisionado; JupyterLab; TensorFlow/Keras apenas nos laboratórios indicados.

**Escopo de validação:** os exemplos básicos podem ser reproduzidos sem TensorFlow; laboratórios opcionais precisam de validação local independente.

---

**Deep Learning · Capítulo de estudo independente**

> Material didático autoral · estudo independente · exemplos locais e laboratórios opcionais

**Referência curricular:** `material histórico de referência`. O material original foi preservado; este capítulo é uma reconstrução didática independente.

### Materiais complementares

- [Guia Técnico](./reference_materials/11_regularizacao_callbacks.pdf)
- [Cheat Sheet](./cheat_sheets/cheat_sheet_regularizacao_callbacks.pdf)


## Como estudar este capítulo

1. Leia a teoria e deduza as expressões apresentadas.
2. Execute as células do **núcleo obrigatório** em um kernel limpo.
3. Confira a interpretação e responda ao exercício antes de abrir a solução.
4. Se tiver TensorFlow e Keras compatíveis, execute o **laboratório opcional**.

**Pré-requisitos transversais:** Python (01), Estatística (03), preparação/validação de ML (05); aplicações textuais remetem à PLN (06). Não há necessidade de repetir integralmente esses cursos.

## Objetivos de aprendizagem

- Identificar e combater overfitting com métodos apropriados.
- Relacionar a técnica à sua função de perda, representação e protocolo de avaliação.
- Explicar limitações e formular um teste reprodutível.

## 1. Fundamentação conceitual

L2 penaliza magnitude de pesos, dropout desativa unidades aleatoriamente durante treinamento, BatchNorm normaliza ativações por lote; EarlyStopping monitora validação. Cuidado com conjuntos de validação pequenos e parada prematura.

**Fonte histórica de apoio:** notas do curso original, consultadas apenas como referência de contexto; as explicações e exemplos desta edição são independentes.

**Questões críticas:** quais pressupostos são necessários? O que a função aprende? Qual métrica indica utilidade real? Que tipo de vazamento ou má interpretação invalidaria a conclusão?

## 2. Matemática essencial

**Expressão esquemática:** `L_total=L_dados+\lambda ||W||²`

A expressão acima resume a relação central. Para a derivação, identifique dimensões, hipóteses e operações; não interprete notação compacta como uma implementação completa.

## 3. Experimento obrigatório — NumPy e/ou scikit-learn

Este experimento é propositalmente pequeno: **executa sem GPU, internet ou serviço externo**. Quando há treino de baseline, não interpretá-lo como evidência da qualidade de uma rede neural.

In [1]:
import numpy as np
rng=np.random.default_rng(12)
W=np.array([1.,-2.,.5]);lam=.1
pen=lam*np.sum(W**2)
mask=(rng.random(10000)>.2).astype(int)/.8
print('L2=',pen,'dropout média do multiplicador=',round(mask.mean(),3))
assert abs(mask.mean()-1)<.06

L2= 0.525 dropout média do multiplicador= 1.004


### Interpretação e limitações

Compare o resultado obtido com a formulação matemática. Confirme shapes, escala e hipótese de modelagem. **Um experimento sintético valida uma operação, não substitui avaliação em dados reais.** Ao treinar modelos, separe dados e ajuste transformações apenas no treino; preserve um conjunto final de teste.

## 4. Laboratório neural opcional — TensorFlow / Keras 3

Esta seção demonstra **APIs reais**, mas depende de versões compatíveis e pode exigir mais recursos que o núcleo. Não é necessária para executar o notebook básico. Não se deve confundir um teste de chamada de API com avaliação experimental de desempenho. Se o pacote TensorFlow não estiver disponível, será exibido um aviso de não execução.

In [2]:
# Laboratório opcional: execute após instalar e configurar TensorFlow/Keras compatíveis.
# Se TensorFlow estiver ausente, o exemplo é ignorado com aviso explícito.
import importlib.util
if importlib.util.find_spec('tensorflow') is None:
    print('Laboratório TensorFlow/Keras não executado: TensorFlow ausente neste kernel.')
else:
    import keras
    from keras import layers
    rede=keras.Sequential([keras.Input(shape=(8,)),layers.Dense(16,kernel_regularizer=keras.regularizers.l2(1e-3)),layers.BatchNormalization(),layers.Activation('relu'),layers.Dropout(.2),layers.Dense(1)])
    rede.compile(optimizer='adam',loss='mse')
    callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss',patience=3,restore_best_weights=True)];print(rede.count_params(),callbacks)

Laboratório TensorFlow/Keras não executado: TensorFlow ausente neste kernel.


## 5. Exercício de fixação

**Questão:** Por que o Dropout é normalmente desativado na inferência?

<details>
<summary>Confira a solução comentada</summary>

**Solução:** Porque o modelo deve produzir previsões estáveis usando a rede completa; a escala é ajustada no treinamento.

**Verificação:** explique o resultado usando notação, shapes e consequências para generalização.
</details>

## 6. Erros comuns e boas práticas

- Confundir ajuste no treino com generalização para dados não vistos.
- Reportar somente uma métrica ou uma execução aleatória.
- Ignorar shape, tipo de variável e convenção do backend.
- Executar exemplos com pesos baixados sem registrar versão, licença e procedência.
- Confundir demonstração matemática com validação de um modelo de produção.

## 7. Resumo de consulta rápida

**Ideia central:** L2 penaliza magnitude de pesos, dropout desativa unidades aleatoriamente durante treinamento, BatchNorm normaliza ativações por lote; EarlyStopping monitora validação.

**Expressão:** `L_total=L_dados+λ||W||²`.

**Lembrete:** começar por baseline; respeitar separação de dados; documentar decisões; avaliar limitações.

**Ponte com outros cursos:** ML I (05) para métricas e validação; PLN (06) para modelos de linguagem; projetos multidisciplinares ficam reservados à fase posterior.

**Fontes de atualização (consultar quando necessário):** [Keras 3](https://keras.io/guides/), [TensorFlow](https://www.tensorflow.org/guide), [scikit-learn](https://scikit-learn.org/stable/user_guide.html).

In [3]:
import numpy as np
rng=np.random.default_rng(42)
x=np.ones(10000)
mascara=rng.binomial(1,.7,size=x.size)/.7
print('Dropout invertido: média aproximada:',round(float(np.mean(x*mascara)),3))
print('Importante: dropout é desligado na inferência')

Dropout invertido: média aproximada: 1.008
Importante: dropout é desligado na inferência


## Laboratório aplicado revisado — Regularização e early stopping em treinamento controlado

Regularização de pesos, dropout e early stopping têm mecanismos distintos. A validação para interrupção deve ficar dentro do treino.

**Procedimento:** execute a próxima célula com o núcleo declarado em `requirements.txt`, confira os `assert` e interprete os resultados. O experimento é independente dos exemplos anteriores.

In [4]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import numpy as np
X,y=make_moons(n_samples=300,noise=.25,random_state=13)
Xtr,Xte,ytr,yte=train_test_split(X,y,stratify=y,test_size=.3,random_state=13)
m=MLPClassifier(hidden_layer_sizes=(32,16),alpha=.05,early_stopping=True,validation_fraction=.2,n_iter_no_change=12,max_iter=400,random_state=13)
p=make_pipeline(StandardScaler(),m).fit(Xtr,ytr)
print('Épocas efetivas:',m.n_iter_,'melhor escore validação:',m.best_validation_score_)
print('Acurácia teste:',round(p.score(Xte,yte),3));assert m.n_iter_<=400

Épocas efetivas: 52 melhor escore validação: 0.9047619047619048
Acurácia teste: 0.844


### Exportação de resultados (local)

Execute esta célula após o experimento acima. Os arquivos são salvos em `outputs/` e não entram automaticamente no Git.

In [5]:
# Exportação do ensaio de regularização sobre a partição já definida.
from pathlib import Path
import pandas as pd

out=Path('outputs'); out.mkdir(exist_ok=True)
resultado=pd.DataFrame([{'modelo':'MLPClassifier','alpha':float(m.alpha),
                         'epocas_efetivas':int(m.n_iter_),
                         'acuracia_teste':float(p.score(Xte,yte))}])
caminho=out/'11_regularizacao_resultado.csv'
resultado.to_csv(caminho,index=False)
print('Exportado:',caminho)


Exportado: outputs/11_regularizacao_resultado.csv


### Interpretação, limites e verificação

O early stopping envolve uma divisão de validação dentro do treino; não utilizar o teste para decidir quando parar.

**Checklist:** conferir entradas e saídas, sementes, inexistência de uso dos dados de teste durante ajuste e distinção entre uma demonstração sintética e desempenho em aplicações reais.